# Notebook 5: Defense Mechanisms

Demonstrate all 8 defense mechanisms against adversarial attacks.

**Defenses:** Input Sanitization, Temporal Consistency, Multi-Sensor Agreement, Robust Clustering, Anomaly Detection, Certified/Randomized Smoothing, Adversarial Training, Ensemble

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from helpers import (load_scenario, get_all_detections, get_ground_truth,
                     CameraAttackerDF, PointCloudAttackerDF, FusionAttackerDF,
                     DefensePipelineDF, CertifiedDefenseDF, AdversarialTrainingDF)
from attacks.camera_attacks import AttackType
from attacks.radar_lidar_attacks import PointCloudAttackType
from attacks.fusion_attacks import FusionAttackType
from defenses.defense_mechanisms import DefenseType

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)

## 5.1 Load Data & Create Attacks

In [ ]:
SCENARIO = 'scenario2'
loader = load_scenario(SCENARIO)
detections = get_all_detections(loader)
ground_truth = get_ground_truth(loader)

# Camera attack
cam_attacker = CameraAttackerDF(epsilon=0.05, num_steps=10)
ir_attacked = cam_attacker.attack_detections(detections[3].copy(), AttackType.FGSM, sensor_id=3)

# Point cloud attack
pc_attacker = PointCloudAttackerDF(epsilon=5.0)
lidar_attacked = pc_attacker.attack_detections(detections[1].copy(), PointCloudAttackType.GHOST_INJECTION, sensor_id=1)

# Fusion attack
fus_attacker = FusionAttackerDF()
fusion_attacked = fus_attacker.attack_scenario(detections.copy(), FusionAttackType.SENSOR_DOS, ground_truth)

print('Attacks created successfully')

## 5.2 Defense Pipeline

In [ ]:
pipeline = DefensePipelineDF()

# Test each defense type
defense_types = [
    DefenseType.INPUT_SANITIZATION,
    DefenseType.TEMPORAL_CONSISTENCY,
    DefenseType.MULTI_SENSOR_AGREEMENT,
    DefenseType.ROBUST_CLUSTERING,
    DefenseType.ANOMALY_DETECTION,
    DefenseType.CERTIFIED,
    DefenseType.ADVERSARIAL_TRAINING,
    DefenseType.ENSEMBLE
]

defended_results = {}
for dtype in defense_types:
    defended = pipeline.defend_detections({'3': ir_attacked.copy()}, dtype, ground_truth)
    defended_results[dtype.name] = defended
    print(dtype.name + ':', len(defended.get('3', pd.DataFrame())), 'detections')

## 5.3 Certified Defense

In [ ]:
cert_def = CertifiedDefenseDF(certified_radius=0.05)
cert_defended = cert_def.defend({'3': ir_attacked.copy()}, ground_truth)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

ax1.hist(detections[3]['bearing'].dropna(), bins=50, alpha=0.5, label='Benign')
ax1.set_title('Benign IR Camera')
ax1.set_xlabel('Bearing (rad)')
ax1.legend()
ax1.grid(True)

ax2.hist(ir_attacked['bearing'].dropna(), bins=50, alpha=0.5, label='Attacked', color='red')
ax2.set_title('FGSM Attacked')
ax2.set_xlabel('Bearing (rad)')
ax2.legend()
ax2.grid(True)

ax3.hist(cert_defended['3']['bearing'].dropna(), bins=50, alpha=0.5, label='Certified Defense', color='green')
ax3.set_title('Certified Defense')
ax3.set_xlabel('Bearing (rad)')
ax3.legend()
ax3.grid(True)

plt.tight_layout()
plt.show()

## 5.4 Adversarial Training

In [ ]:
adv_train = AdversarialTrainingDF(augmentation_ratio=0.3)
adv_train.train({'3': detections[3].copy()}, {'3': ir_attacked.copy()})
adv_defended = adv_train.defend({'3': ir_attacked.copy()})

print('Adversarial training defense applied')
print('Original attacked:', len(ir_attacked))
print('After defense:', len(adv_defended['3']))

## 5.5 Defense Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

names = list(defended_results.keys())
counts = [len(d.get('3', pd.DataFrame())) for d in defended_results.values()]

bars = ax.bar(range(len(names)), counts, color='steelblue')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_ylabel('Detection Count')
ax.set_title('Defense Comparison: IR Camera Detections After FGSM Attack')
ax.axhline(y=len(detections[3]), color='green', linestyle='--', label='Benign count')
ax.axhline(y=len(ir_attacked), color='red', linestyle='--', label='Attacked count')
ax.legend()
ax.grid(True, axis='y')
plt.tight_layout()
plt.show()